In [1]:
using LinearAlgebra
using BSplineKit
using PyCall
using DelimitedFiles
using Plots
using NonlinearEigenproblems
include("BaseFlow_cavity.jl")
include("Stability_Cavity.jl")

mode = 1:cavity; mode = 2:stationary; mode = 3:rotation;


Main.CRC_STA

In [24]:
Res = 2000
N_cheb = 129
mode = 1
Ro = - 1.0
Co = 2 - Ro - Ro^2
Ts = -0.4
u0,v0,w0,du0,dv0,x = CRC_BF.BaseFlow(Res,Ro,Ts,mode)
D,D2,z = CRC_BF.Cheb(N_cheb,mode)
F,G,H = CRC_BF.interp(u0,v0,w0,z,N_cheb,mode)

([-4.4575275345863566e-23; 0.003229083230533703; … ; -0.0007629751460330066; 1.1314796803959727e-20;;], [1.0; 0.9971524857918574; … ; 0.000627115232262192; -1.0044816069749407e-20;;], [0.4; 0.39997855621288875; … ; -5.0619515254512985e-6; -1.1937870600365295e-20;;])

In [30]:
R = 600
be =  0.3
alpha_r = 0.255
alpha_i = -0.14
alpha = alpha_r + alpha_i * im
cof = CRC_STA.Spatial_mode_BEK1((F),(G.-1),(H),R,N_cheb,D,D2,Res)
# val,vec = EigenCore(cof,D,D2,be,alpha,R,N_cheb)
H0,H1 = CRC_STA.assemble_time_mat(cof,D,D2,be,alpha,R,N_cheb)
C = eigen(H0,H1)
val = C.values
vec = C.vectors
map_index0 = map(x-> abs(real(x)) < 0.1 && abs(imag(x)) < 0.1, val)
val_filter0 = val[map_index0]
vec_filter0 = vec[:,map_index0]
addition = 0.017
indictor = sum(real(val_filter0))/length(val_filter0)
map_index = map(x-> (real(x)) > (indictor) + addition , val_filter0)
val_filter1 = val_filter0[map_index]
vec_filter1 = vec_filter0[:,map_index]
val_filter2 = val_filter1[findmax(imag.(val_filter1))[2]]
vec_filter2 = vec_filter1[:,findmax(imag.(val_filter1))[2]];
val_filter = val_filter2
vec_filter = vec_filter2
val_target = val_filter
vec_target = vec_filter
val_target

-0.02356942782601064 - 0.043048154638450145im

In [31]:
(indictor) + addition

-0.04469266422256801

In [ ]:
scatter(real(val),imag(val),xlims=[-0.5,0.5],ylims=[-0.5,0.5])

In [ ]:
scatter(real(val_filter0),imag(val_filter0))

In [ ]:
val = val_target[1] 
vec = vec_target[:,1]
dir = 1
writedlm("test.dat",[])
for alpha_r = 0.2 : 0.0025 : 0.3
    for alpha_i = -0.1: -0.0025 : -0.2
        alpha = alpha_r + alpha_i * im
        H0,H1 = CRC_STA.assemble_time_mat(cof,D,D2,be,alpha,R,N_cheb)
        val,vec = rayleigh_quotient_iteration(H0, H1, val+ 0.01im, vec)
        open("test.dat","a") do io
            writedlm(io,[R alpha_r alpha_i be real(val) imag(val)])
        end
    end
end

In [5]:
function rayleigh_quotient_iteration(A, B, sigma, q0=rand(ComplexF64, size(A, 1)))
    tol = 1e-8
    sigma_current = ComplexF64(sigma)
    q = q0 / norm(q0) # 必须使用欧几里得范数
    
    for i in 1:20
        sigma_old = sigma_current
        
        v = (A - sigma_current * B) \ (B * q)
        q = v / norm(v)
        
        num = dot(q, A * q)
        den = dot(q, B * q)
        sigma_current = num / den
        
        if abs(sigma_current - sigma_old) < tol
            return sigma_current, q
        end
    end
    @warn "RQI failed to converge"
    return sigma_current, q
end

rayleigh_quotient_iteration (generic function with 2 methods)

In [38]:
function compute_omega(alpha, be, R, cof, D, D2)

    A_mat,B_mat = CRC_STA.assemble_time_mat(cof,D,D2,be,alpha,R,N_cheb)
    eig_result = eigen(A_mat, B_mat)
    val = eig_result.values
    vec = eig_result.vectors
    # 找到虚部最大 (最不稳定) 的特征值
    map_index0 = map(x-> abs(real(x)) < 0.2 && abs(imag(x)) < 0.1, val)
    val_filter0 = val[map_index0]
    vec_filter0 = vec[:,map_index0]
    # addition = 0.005
    addition = 0.02 - 0.05 * be
    indictor = sum(real(val_filter0))/length(val_filter0)
    map_index = map(x-> (real(x)) > (indictor) + addition , val_filter0)
    val_filter1 = val_filter0[map_index]
    vec_filter1 = vec_filter0[:,map_index]
    val_filter2 = val_filter1[findmax(imag.(val_filter1))[2]]
    vec_filter2 = vec_filter1[:,findmax(imag.(val_filter1))[2]];

    return val_filter2,vec_filter2
end

compute_omega (generic function with 1 method)

In [20]:
function Newton_Raphson(alpha, be, R, cof, D, D2, N_cheb)
    delta_alpha = 1e-5
    val,vec = compute_omega(alpha, be, R, cof, D, D2)
    H0_forward,H1_forward = CRC_STA.assemble_time_mat(cof,D,D2,be,alpha + delta_alpha,R,N_cheb)
    H0_backward,H1_backward = CRC_STA.assemble_time_mat(cof,D,D2,be,alpha - delta_alpha,R,N_cheb)
    val_forward,vec_forward = rayleigh_quotient_iteration(H0_forward, H1_forward, val, vec)
    val_backward,vec_backward = rayleigh_quotient_iteration(H0_backward, H1_backward, val, vec)
    diff1 = (val_forward - val_backward) / (2*delta_alpha)
    diff2 = (val_forward - 2*val + val_backward) / (delta_alpha^2)
    alpha_new = alpha - diff1/diff2
    tol = abs(diff1)
    return alpha_new, tol
end

Newton_Raphson (generic function with 1 method)

In [8]:
function ab_eigen(R,be,alpha_ini)
    alpha = alpha_ini
    cof = CRC_STA.Spatial_mode_BEK1((F),(G.-1),(H),R,N_cheb,D,D2,Res)
    tol = 1
    while abs(tol) > 1e-8
        alpha_new, tol = Newton_Raphson(alpha, be, R, cof, D, D2, N_cheb)
        alpha = alpha_new
        open("test_1.dat","a") do io
            writedlm(io,[alpha_new tol])
        end 
    end
    val,vec = compute_omega(alpha, be, R, cof, D, D2)
    return alpha, val
end

ab_eigen (generic function with 1 method)

In [25]:
Res = 2000
N_cheb = 129
mode = 1
Ro = - 1.0
Co = 2 - Ro - Ro^2
Ts = -0.4
u0,v0,w0,du0,dv0,x = CRC_BF.BaseFlow(Res,Ro,Ts,mode)
D,D2,z = CRC_BF.Cheb(N_cheb,mode)
F,G,H = CRC_BF.interp(u0,v0,w0,z,N_cheb,mode)

([-4.4575275345863566e-23; 0.003229083230533703; … ; -0.0007629751460330066; 1.1314796803959727e-20;;], [1.0; 0.9971524857918574; … ; 0.000627115232262192; -1.0044816069749407e-20;;], [0.4; 0.39997855621288875; … ; -5.0619515254512985e-6; -1.1937870600365295e-20;;])

In [46]:
writedlm("AS_$(Ts).dat",[])
writedlm("eigen.dat",[])
R = 600
be =  0.28
beta_delta = -0.001
R_delta = 2
alpha = 0.2 - 0.1 * im
alpha,val = ab_eigen(R,be,alpha)
open("eigen.dat","a") do io
    writedlm(io,[R be alpha val])
end 
dir_beta = -1 * sign(imag(val))
while abs(imag(val)) > 8e-5
    be += dir_beta * beta_delta
    alpha,val = ab_eigen(R,be,alpha)
    open("eigen.dat","a") do io
        writedlm(io,[R be alpha val])
    end 
end
open("AS_$(Ts).dat","a") do io
    writedlm(io,[R be real(alpha) imag(alpha) real(val) imag(val)])
end
be_ini = be
alpha_ini = alpha

while R <= 620

    alpha,val = ab_eigen(R,be + beta_delta,alpha)
    dir_R = -1 * sign(imag(val))
    val_ori = val
    open("eigen.dat","a") do io
        writedlm(io,[R be alpha val])
    end 
    while imag(val) * imag(val_ori) > 0
        R_delta = 2
        beta_delta = -0.0025
        R += dir_R * R_delta
        alpha,val = ab_eigen(R,be,alpha)
        open("eigen.dat","a") do io
            writedlm(io,[R be alpha val])
        end 
    end
    open("AS_$(Ts).dat","a") do io 
    writedlm(io,[R be real(alpha) imag(alpha) real(val) imag(val)])
    end
    be += beta_delta
end